# Qwen3.8-Flash-Next on Kaggle (Strata Gyro-S)

Serves Gyro-S on this notebook's GPUs and prints an OpenAI-compatible URL for Cursor.

Before you run:

1. Settings → Accelerator → **GPU T4 x2**.
2. Settings → Internet → **On**.
3. Optional: attach a private dataset that already contains `Qwen3.8-Flash-Next-Gyro-S-TQ1_0.gguf`. The notebook symlinks it and does not make a second 58 GB copy.

The last cell keeps the session alive. The Cloudflare address and API key change every start. They are a secret for your own coding client. Do not share the notebook output.

Kaggle ends the session after 12 hours, or sooner when the GPU quota runs out.




In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

MIN_COMBINED_GIB = 27.6
MIN_DOWNLOAD_GIB = 65.0
MIN_ATTACHED_GIB = 12.0
MIN_HOME_GIB = 8.0
GGUF_NAME = "Qwen3.8-Flash-Next-Gyro-S-TQ1_0.gguf"


def gib(num_bytes):
    return num_bytes / (1024 ** 3)


def mem_total_bytes():
    for line in Path("/proc/meminfo").read_text().splitlines():
        if line.startswith("MemTotal:"):
            return int(line.split()[1]) * 1024
    raise RuntimeError("MemTotal is missing from /proc/meminfo")


def gpu_rows():
    if shutil.which("nvidia-smi") is None:
        raise RuntimeError("nvidia-smi is not on PATH. Turn on the T4 x2 accelerator and rerun.")
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=index,name,memory.total,driver_version",
            "--format=csv,noheader,nounits",
        ],
        check=True,
        text=True,
        capture_output=True,
    )
    rows = []
    for line in result.stdout.splitlines():
        if not line.strip():
            continue
        index, name, total_mib, driver = [part.strip() for part in line.split(",")]
        rows.append(
            {
                "index": int(index),
                "name": name,
                "vram_bytes": int(float(total_mib)) * 1024 ** 2,
                "driver": driver,
            }
        )
    if not rows:
        raise RuntimeError("No GPU is visible. Turn on GPU T4 x2 and rerun.")
    return rows


def mounts():
    found = []
    for line in Path("/proc/mounts").read_text().splitlines():
        parts = line.split()
        if len(parts) < 3:
            continue
        _device, point, fstype = parts[0], parts[1], parts[2]
        if fstype in {"tmpfs", "devtmpfs", "proc", "sysfs", "squashfs", "overlayfs"}:
            continue
        if point.startswith("/proc") or point.startswith("/sys") or point.startswith("/dev"):
            continue
        if point.startswith("/kaggle/input"):
            continue
        try:
            usage = shutil.disk_usage(point)
        except OSError:
            continue
        probe = Path(point) / ".strata_write_probe"
        try:
            probe.write_text("ok")
            probe.unlink()
        except OSError:
            continue
        found.append((point, usage.free))
    return found


def find_attached_gguf():
    root = Path("/kaggle/input")
    if not root.is_dir():
        return None
    matches = sorted(root.rglob(GGUF_NAME))
    if matches:
        return matches[0]
    matches = sorted(root.rglob("*Gyro-S*.gguf"))
    return matches[0] if matches else None


gpus = gpu_rows()
ram_bytes = mem_total_bytes()
vram_bytes = sum(gpu["vram_bytes"] for gpu in gpus)
combined = gib(vram_bytes + ram_bytes)
driver = gpus[0]["driver"]
driver_major = int(driver.split(".")[0])

print("GPUs")
for gpu in gpus:
    print(f"  [{gpu['index']}] {gpu['name']}  {gib(gpu['vram_bytes']):.1f} GiB  driver {gpu['driver']}")
print(f"System RAM: {gib(ram_bytes):.1f} GiB")
print(f"VRAM + RAM: {combined:.1f} GiB (Gyro-S weights need about {MIN_COMBINED_GIB:.1f} GiB combined)")
print()
print("Writable disks")
disk_rows = sorted(mounts(), key=lambda item: item[1], reverse=True)
for point, free in disk_rows:
    print(f"  {gib(free):6.1f} GiB free  {point}")

if combined < MIN_COMBINED_GIB:
    raise RuntimeError(
        f"VRAM + RAM is {combined:.1f} GiB. Gyro-S needs about {MIN_COMBINED_GIB:.1f} GiB combined. Stopping."
    )

attached = find_attached_gguf()
need_gib = MIN_ATTACHED_GIB if attached else MIN_DOWNLOAD_GIB
candidates = [(point, free) for point, free in disk_rows if gib(free) >= need_gib]
if not candidates:
    if attached:
        raise RuntimeError(
            f"The attached GGUF still needs about {MIN_ATTACHED_GIB:.0f} GiB free for Strata's pack and draft. "
            "No writable disk has that much space."
        )
    raise RuntimeError(
        f"No writable disk has {MIN_DOWNLOAD_GIB:.0f} GiB free, and no Gyro-S GGUF is attached under /kaggle/input. "
        "Attach a private dataset that contains the file, or use a session disk with at least 65 GiB free. "
        "/kaggle/working is only 20 GiB, so the weights cannot go there."
    )

# Prefer a disk other than the 20 GiB working folder when another disk qualifies.
data_mount = candidates[0][0]
for point, _free in candidates:
    if point != "/kaggle/working":
        data_mount = point
        break

def mount_of(path):
    path = str(path)
    best = ""
    for point, _free in disk_rows:
        if path == point or path.startswith(point.rstrip("/") + "/"):
            if len(point) > len(best):
                best = point
    return best


home_mount = mount_of(Path.home())
home_free = gib(shutil.disk_usage(Path.home()).free)
if home_mount and home_mount != data_mount and home_free < MIN_HOME_GIB:
    raise RuntimeError(
        f"The home disk ({home_mount}) has {home_free:.1f} GiB free. Strata's engine needs about {MIN_HOME_GIB:.0f} GiB there."
    )

data_dir = Path(data_mount) / "strata-gyro-s"
data_dir.mkdir(parents=True, exist_ok=True)
model_path = None
if attached:
    link = data_dir / GGUF_NAME
    if link.is_symlink() or link.exists():
        link.unlink()
    link.symlink_to(attached)
    model_path = link
    print()
    print(f"Using attached weights via symlink (no second copy): {attached}")
else:
    print()
    print(f"No attached GGUF. Gyro-S will download into {data_dir}")

print(f"Data dir: {data_dir} ({gib(shutil.disk_usage(data_dir).free):.1f} GiB free)")
print(f"CUDA package: {'12' if driver_major < 580 else 'default'} (driver {driver})")

os.environ["STRATA_DATA_DIR"] = str(data_dir)
os.environ["STRATA_MODEL_PATH"] = str(model_path or "")
os.environ["STRATA_GPU_LIST"] = ",".join(str(gpu["index"]) for gpu in gpus)
os.environ["STRATA_USE_CUDA12"] = "1" if driver_major < 580 else "0"




In [ ]:
import os
import shutil
import stat
import subprocess
import urllib.request
from pathlib import Path

bindir = Path.home() / ".local" / "bin"
bindir.mkdir(parents=True, exist_ok=True)
os.environ["PATH"] = str(bindir) + os.pathsep + os.environ.get("PATH", "")

def download(url, dest):
    request = urllib.request.Request(url, headers={"User-Agent": "qwen-flash-next-kaggle"})
    with urllib.request.urlopen(request) as response, open(dest, "wb") as out:
        shutil.copyfileobj(response, out)


cloudflared = bindir / "cloudflared"
if not cloudflared.exists():
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    print(f"Downloading cloudflared from {url}")
    download(url, cloudflared)
    cloudflared.chmod(cloudflared.stat().st_mode | stat.S_IEXEC)

strata = bindir / "strata"
if shutil.which("strata") is None:
    script = Path("/tmp/strata-install.sh")
    print("Installing the Strata terminal")
    download(
        "https://github.com/Niko1221/Strata/releases/latest/download/install.sh",
        script,
    )
    subprocess.run(["sh", str(script)], check=True)
    if shutil.which("strata") is None and not strata.exists():
        raise RuntimeError(
            "The Strata installer finished but `strata` is not on PATH. "
            f"Looked in {bindir}."
        )

print("cloudflared:", subprocess.check_output([str(cloudflared), "--version"], text=True).strip())
strata_bin = shutil.which("strata") or str(strata)
print("strata:", strata_bin)
version = subprocess.run([strata_bin, "--version"], text=True, capture_output=True)
print((version.stdout or version.stderr).strip())
os.environ["STRATA_BIN"] = strata_bin




In [ ]:
import os
import re
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path

STRATA = os.environ["STRATA_BIN"]
DATA_DIR = os.environ["STRATA_DATA_DIR"]
MODEL_PATH = os.environ.get("STRATA_MODEL_PATH") or ""
GPU_LIST = os.environ.get("STRATA_GPU_LIST", "0")
USE_CUDA12 = os.environ.get("STRATA_USE_CUDA12") == "1"
LOG_PATH = Path("/tmp/strata-kaggle.log")
PORT = "8080"
URL_RE = re.compile(r"https://[A-Za-z0-9-]+\.trycloudflare\.com")
KEY_RE = re.compile(r"(?:api[_ ]?key|key)\s*[:=]\s*(\S+)", re.IGNORECASE)
MEMORY_MARKERS = (
    "out of memory",
    "cannot allocate",
    "bad_alloc",
    "killed",
    "pinned-memory-exhausted",
    "cuda error: out of memory",
)


def environment():
    env = os.environ.copy()
    bindir = str(Path.home() / ".local" / "bin")
    env["PATH"] = bindir + os.pathsep + env.get("PATH", "")
    return env


def available_ram_gib():
    for line in Path("/proc/meminfo").read_text().splitlines():
        if line.startswith("MemAvailable:"):
            return int(line.split()[1]) / (1024 ** 2)
    return 0.0


def run_stream(cmd):
    print("+", " ".join(cmd), flush=True)
    with LOG_PATH.open("a", encoding="utf-8") as log:
        log.write("\n$ " + " ".join(cmd) + "\n")
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=environment(),
            text=True,
            bufsize=1,
        )
        chunks = []
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
            chunks.append(line)
        code = proc.wait()
    text = "".join(chunks)
    return code, text


def launch_cmd(retry):
    cmd = [
        STRATA,
        "launch",
        "--context",
        "32k",
        "--vision",
        "off",
        "--parallel",
        "1",
        "--gpu",
        GPU_LIST,
        "--access",
        "cloudflare",
        "--port",
        PORT,
        "--new-key",
        "--yes",
        "--no-tui",
        "--data-dir",
        DATA_DIR,
        "--set",
        "expert-cache=auto",
    ]
    if MODEL_PATH:
        cmd.extend(["--model-path", MODEL_PATH])
    else:
        cmd.extend(["--family", "gyro", "--model", "Gyro-S"])
    if USE_CUDA12:
        cmd.extend(["--set", "STRATA_CUDA=12"])
    if retry:
        cmd.extend(
            [
                "--restart",
                "--set",
                "resident-budget-gib=4",
                "--set",
                "STRATA_RESIDENT_PIN=0",
            ]
        )
    return cmd


def looks_like_memory(text):
    lowered = text.lower()
    return any(marker in lowered for marker in MEMORY_MARKERS)


def server_up():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=5) as response:
            return 200 <= response.status < 500
    except urllib.error.HTTPError as exc:
        return exc.code in {401, 403}
    except (urllib.error.URLError, TimeoutError, OSError):
        return False


def wait_until_up(seconds):
    deadline = time.time() + seconds
    while time.time() < deadline:
        if server_up():
            return True
        time.sleep(15)
    return False


def capture(args):
    result = subprocess.run(
        [STRATA, *args],
        check=False,
        text=True,
        capture_output=True,
        env=environment(),
    )
    return (result.stdout or "") + (result.stderr or "")


def find_url(*blobs):
    for blob in blobs:
        found = URL_RE.findall(blob or "")
        if found:
            return found[-1]
    return ""


def find_key(key_text):
    text = (key_text or "").strip()
    match = KEY_RE.search(text)
    if match:
        return match.group(1).strip().strip('"').strip("'")
    for line in reversed([line.strip() for line in text.splitlines() if line.strip()]):
        token = line.strip('"').strip("'")
        if re.fullmatch(r"[A-Za-z0-9_\-]{16,}", token):
            return token
    return ""


def find_model(*blobs):
    for blob in blobs:
        if not blob:
            continue
        match = re.search(r"(gyro-s[A-Za-z0-9_.-]*)", blob, re.IGNORECASE)
        if match:
            return match.group(1)
    return "gyro-s-strata"


print("Launch plan (no download yet)")
plan_cmd = [STRATA, "launch", "--plan", "--context", "32k", "--vision", "off", "--data-dir", DATA_DIR]
if MODEL_PATH:
    plan_cmd.extend(["--model-path", MODEL_PATH])
else:
    plan_cmd.extend(["--family", "gyro", "--model", "Gyro-S"])
plan_code, plan_text = run_stream(plan_cmd)
if "couldn't find an engine" in plan_text.lower():
    raise RuntimeError(
        "Strata has no engine for this GPU. The T4 is Turing (sm_75). "
        "If the log above names a driver or CUDA mismatch, that is the blocker."
    )
if plan_code != 0:
    print(f"Plan exited {plan_code}. Continuing to launch; Strata uses the plan as a warning.")

attempt_text = ""
for attempt in (1, 2):
    retry = attempt == 2
    if retry:
        print(
            "\nFirst start did not stay up under memory pressure. "
            "Retrying once with a 4 GiB resident expert budget.\n"
        )
    code, text = run_stream(launch_cmd(retry))
    attempt_text += "\n" + text
    # A clean return can still be loading the engine. A failed return should not sit for 20 minutes.
    up = server_up() or wait_until_up(20 * 60 if code == 0 else 45)
    if up:
        break
    ram_free = available_ram_gib()
    memory_pressure = looks_like_memory(text) or ram_free < 1.5
    if attempt == 1 and memory_pressure:
        print(
            f"Server is not accepting requests (exit {code}, {ram_free:.1f} GiB RAM free). "
            "The machine is at its RAM ceiling."
        )
        subprocess.run([STRATA, "launch", "--stop"], check=False, env=environment())
        continue
    raise RuntimeError(
        "Strata did not start. The log is /tmp/strata-kaggle.log. "
        "If this is an out-of-memory or T4 kernel failure, use OpenRouter's hosted qwen/qwen3.8-flash instead."
    )

status_text = ""
public = ""
for _ in range(18):
    status_text = capture(["launch", "--status"])
    public = find_url(attempt_text, status_text, LOG_PATH.read_text(encoding="utf-8", errors="replace"))
    if public:
        break
    time.sleep(10)
key_text = capture(["launch", "--show-key"])
api_key = find_key(key_text)
model_id = find_model(status_text, attempt_text)
if not public:
    raise RuntimeError(
        "The server is up on 127.0.0.1:8080 but no Cloudflare URL was printed. "
        "Check /tmp/strata-kaggle.log. cloudflared has to be on PATH before launch."
    )

base_url = public.rstrip("/") + "/v1"
print("\n" + "=" * 72)
print("Copy this into .env and into Cursor. Do not share it.")
print(f"STRATA_BASE_URL={base_url}")
print(f"STRATA_API_KEY={api_key or '(run: strata launch --show-key)'}")
print(f"STRATA_MODEL={model_id}")
print("Local check: http://127.0.0.1:8080/v1")
print("Available RAM now: {:.1f} GiB".format(available_ram_gib()))
print("Leave this cell running. The model stops when the cell or the Kaggle session stops.")
print("=" * 72 + "\n")

try:
    while True:
        time.sleep(300)
        if server_up():
            print(time.strftime("%H:%M:%S"), "serving", flush=True)
        else:
            print(time.strftime("%H:%M:%S"), "local port closed; stopping keep-alive", flush=True)
            print(capture(["launch", "--status"]))
            break
except KeyboardInterrupt:
    print("Stopping Strata")
    subprocess.run([STRATA, "launch", "--stop"], check=False, env=environment())


